# 📊 Dia 1 — Fontes de Dados: .gov e Kaggle
**Data:** 26/05/2026 | **Horário:** 19h – 22h

---

## O que vamos aprender hoje?

1. O que são fontes de dados públicos e o que é uma API
2. Acessar dados reais do governo via **API do IBGE** (estados e regiões)
3. Acessar dados reais do **Banco Central** (inflação IPCA)
4. Navegar o portal **dados.gov.br** e carregar CSVs direto pela URL
5. O que é o **Kaggle** e como baixar datasets
6. Analisar e visualizar tudo com pandas e matplotlib


---
## 🕐 PARTE 1 — O que são Fontes de Dados? (19h – 19h30)

### Fontes de dados são lugares onde encontramos informações organizadas para análise.

| Tipo | Exemplos | Acesso |
|------|----------|--------|
| **Dados Públicos** | IBGE, dados.gov.br, Banco Central | Gratuito |
| **Plataformas de dados** | Kaggle, UCI, Data.world | Gratuito (cadastro) |
| **APIs** | IBGE API, BCB API, INEP | Gratuito |

### O que é uma API?

**API** = *Application Programming Interface* — é uma "porta" que o servidor abre para você buscar dados diretamente pelo código, sem entrar no site manualmente.

```
Você (Python)  →  faz uma requisição  →  Servidor do IBGE
                                               ↓
Você (Python)  ←  recebe os dados   ←  Servidor do IBGE
```

### Por que usar APIs ao invés de baixar arquivos?
- Dados **sempre atualizados** — sem precisar baixar de novo a cada vez
- **Automatiza** a coleta de dados no código
- É a forma profissional de consumir dados públicos


In [1]:
# Importar todas as bibliotecas do dia
import pandas as pd
import matplotlib.pyplot as plt
import requests  # nova! permite acessar URLs e APIs pelo Python

print('Bibliotecas importadas com sucesso!')

Bibliotecas importadas com sucesso!


---
## 🕐 PARTE 2 — Dados do Governo Brasileiro (.gov) (19h30 – 20h30)

### Principais portais e APIs do Brasil:

| Portal | URL | O que tem |
|--------|-----|-----------|
| **dados.gov.br** | dados.gov.br | Portal central — CSVs de todos os ministérios |
| **API IBGE** | servicodados.ibge.gov.br | Estados, municípios, regiões |
| **API Banco Central** | api.bcb.gov.br | Inflação, câmbio, SELIC |
| **INEP** | inep.gov.br | ENEM, IDEB, censo escolar |
| **OpenDataSUS** | opendatasus.saude.gov.br | Saúde pública, vacinação |

---
### Parte 2A — API do IBGE: estados e regiões do Brasil

O IBGE disponibiliza uma API REST **gratuita e sem cadastro** com dados de localidades.

**URL real:**
> `https://servicodados.ibge.gov.br/api/v1/localidades/estados`

Abra esse link no navegador e veja os dados em formato JSON!

In [2]:
# Acessar a API do IBGE — dados reais dos estados brasileiros
url_ibge = 'https://servicodados.ibge.gov.br/api/v1/localidades/estados'

# requests.get() busca o conteúdo da URL (como um navegador)
resposta = requests.get(url_ibge)

print(f'Status da resposta: {resposta.status_code}')  # 200 = OK!
print(f'Tipo dos dados: {type(resposta.json())}')
print(f'Quantidade de estados retornados: {len(resposta.json())}')

Status da resposta: 200
Tipo dos dados: <class 'list'>
Quantidade de estados retornados: 27


In [3]:
# Ver como é um estado no JSON retornado
dados_json = resposta.json()

print('Exemplo — primeiro estado da lista:')
print(dados_json[0])

Exemplo — primeiro estado da lista:
{'id': 11, 'sigla': 'RO', 'nome': 'Rondônia', 'regiao': {'id': 1, 'sigla': 'N', 'nome': 'Norte'}}


In [ ]:

# Transformar o JSON em DataFrame com um laço for
# Cada item da lista é um dicionário com os dados de um estado

lista_estados = []

for estado in dados_json:
    lista_estados.append({
        'Codigo': estado['id'],
        'Sigla':  estado['sigla'],
        'Estado': estado['nome'],
        'Regiao': estado['regiao']['nome'] # acessar o nome da região dentro do dicionário 'regiao'
    })

df_estados = pd.DataFrame(lista_estados)  # Criar DataFrame a partir da lista de dicionários

print(f'{len(df_estados)} estados carregados!')
df_estados.head()


27 estados carregados!


,Codigo,Sigla,Estado,Regiao
0,11,RO,Rondônia,Norte
1,12,AC,Acre,Norte
2,13,AM,Amazonas,Norte
3,14,RR,Roraima,Norte
4,15,PA,Pará,Norte


In [ ]:

# Verificar os dados carregados no DataFrame
print(f'Total de estados: {len(df_estados)}')
print(f'Colunas: {df_estados.columns.tolist()}')
df_estados.head()


Total de estados: 27
Colunas: ['Codigo', 'Sigla', 'Estado', 'Regiao']


,Codigo,Sigla,Estado,Regiao
0,11,RO,Rondônia,Norte
1,12,AC,Acre,Norte
2,13,AM,Amazonas,Norte
3,14,RR,Roraima,Norte
4,15,PA,Pará,Norte


In [ ]:
# Quantos estados por região?
por_regiao = df_estados['Regiao'].value_counts()
print('Estados por Região:')
print(por_regiao)

plt.figure(figsize=(7, 4))
por_regiao.plot(kind='bar', color='steelblue')
plt.title('Estados por Região — Fonte: API IBGE')
plt.ylabel('Número de Estados')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

---
### Parte 2B — API do Banco Central: IPCA (Inflação Oficial)

O **Banco Central do Brasil** possui uma API com séries históricas de indicadores econômicos oficiais.

**IPCA** = Índice de Preços ao Consumidor Amplo — a inflação oficial do Brasil, medida todo mês.

**URL real (com período):**
> `https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json&dataInicial=01/01/2020&dataFinal=31/12/2025`

- `433` = código da série do IPCA
- `dataInicial` / `dataFinal` = período desejado (formato: `dd/mm/aaaa`)

> ⚠️ **Atenção:** A API do BCB limita a **20 registros** no modo `ultimos/N`.  
> Para buscar períodos maiores, usamos `dataInicial` e `dataFinal`.

Abra a URL no navegador e veja o JSON!

In [ ]:
# API do Banco Central — IPCA mensal (Jan/2020 a Dez/2025)
# Usando dataInicial e dataFinal para buscar períodos maiores que 20 registros
url_ipca = (
    'https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados'
    '?formato=json&dataInicial=01/01/2020&dataFinal=31/12/2025'
)

resposta_bcb = requests.get(url_ipca, headers={'Accept': 'application/json'})
print(f'Status: {resposta_bcb.status_code}')  # 200 = OK

df_ipca = pd.DataFrame(resposta_bcb.json())

print(f'Registros carregados: {len(df_ipca)}')
print(f'Colunas: {df_ipca.columns.tolist()}')
df_ipca.head()

In [ ]:

# Ajustar os tipos de dados para trabalhar corretamente
df_ipca['data']  = pd.to_datetime(df_ipca['data'], format='%d/%m/%Y')
df_ipca['valor'] = df_ipca['valor'].astype(float)

print(f'Total de registros: {len(df_ipca)} meses')
df_ipca.head()


In [ ]:

# Estatísticas do IPCA
print('Estatisticas do IPCA (Jan/2020 a Dez/2025):')
print(f'  Media mensal:  {df_ipca["valor"].mean():.2f}%')
print(f'  Maximo mensal: {df_ipca["valor"].max():.2f}%')
print(f'  Minimo mensal: {df_ipca["valor"].min():.2f}%')


In [ ]:

# Gráfico: IPCA ao longo do tempo
plt.figure(figsize=(12, 4))
plt.plot(df_ipca['data'], df_ipca['valor'], color='steelblue', linewidth=1.5)
plt.axhline(y=0, color='red', linestyle='--', linewidth=1)
plt.title('IPCA — Inflacao Mensal (Jan/2020 a Dez/2025) | Fonte: Banco Central do Brasil')
plt.ylabel('Variacao (%)')
plt.xlabel('Data')
plt.tight_layout()
plt.show()


In [ ]:

# IPCA acumulado por ano
df_ipca['Ano'] = df_ipca['data'].dt.year
ipca_por_ano = df_ipca.groupby('Ano')['valor'].sum().round(2)

print('IPCA acumulado por ano (%):')
print(ipca_por_ano)


---
### Parte 2C — dados.gov.br: como navegar e carregar CSVs

O **dados.gov.br** é o portal central de dados abertos do governo federal. Lá você encontra CSVs prontos para baixar ou acessar por URL direta.

### Como encontrar e usar um CSV:

1. Acesse **`dados.gov.br`**
2. Use a barra de busca (ex: `combustíveis`, `educação`, `saúde`)
3. Clique no dataset desejado
4. Procure pelos recursos no formato **CSV**
5. Clique com botão direito no botão de download → **Copiar endereço do link**
6. Use direto no `pd.read_csv(url)` — o pandas lê da internet!

### Dataset real — Preços de Combustíveis (ANP):

> **Página:** `https://dados.gov.br/dados/conjuntos-dados/serie-historica-de-precos-de-combustiveis-e-de-glp`

A ANP (Agência Nacional do Petróleo) publica toda semana os preços de gasolina, diesel e GLP por município.

In [ ]:
# Carregar CSV diretamente da URL do governo — sem baixar o arquivo!
# O pandas aceita URLs diretamente no read_csv()

url_combustivel = (
    'https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/'
    'arquivos/shpc/dsan/ultimas-4-semanas-gasolina-etanol.csv'
)

try:
    df_comb = pd.read_csv(url_combustivel, sep=';', encoding='latin-1')
    print(f'Dataset carregado!')
    print(f'Linhas: {df_comb.shape[0]:,} | Colunas: {df_comb.shape[1]}')
    print()
    print('Colunas:', df_comb.columns.tolist())
except Exception as e:
    print('Não foi possível acessar a URL agora (pode ter mudado).')
    print(f'Erro: {e}')
    print()
    print('Dica: acesse dados.gov.br, busque "combustíveis ANP" e use o link do CSV mais recente.')

In [ ]:

# Se o CSV carregou, vamos explorar os dados!
try:
    print('Primeiras linhas:')
    print(df_comb.head())
    print()
    print(f'Colunas de preco: Valor de Venda, Valor de Compra')
except:
    print('Pule esta célula — o CSV não carregou.')


---
### Parte 2D — Combinando API do IBGE com dados do Censo 2022

A API do IBGE nos deu **estados e regiões**. Agora vamos adicionar **população** do Censo 2022.

> Os dados populacionais do Censo 2022 estão no SIDRA (sistema de tabelas do IBGE):  
> `https://sidra.ibge.gov.br/tabela/9514`

Para a aula, usamos os valores já extraídos do Censo publicado:

In [ ]:
# Dados do Censo IBGE 2022 — por sigla do estado
# Fonte: https://sidra.ibge.gov.br/tabela/9514

populacao_censo2022 = {
    'RO':1815278, 'AC':830026,  'AM':4269995, 'RR':636707,
    'PA':8116132, 'AP':733508,  'TO':1607363, 'MA':6775152,
    'PI':3269200, 'CE':8794957, 'RN':3302406, 'PB':3974495,
    'PE':9058931, 'AL':3127683, 'SE':2298696, 'BA':14136417,
    'MG':20538718,'ES':4064052, 'RJ':16054524,'SP':44420459,
    'PR':11516840,'SC':7609601, 'RS':10882965,'MS':2756700,
    'MT':3784239, 'GO':7206589, 'DF':2817068
}

area_km2 = {
    'RO':237765, 'AC':164123, 'AM':1559167,'RR':224301,
    'PA':1247954,'AP':142470, 'TO':277423, 'MA':329651,
    'PI':251755, 'CE':148826, 'RN':52811,  'PB':56585,
    'PE':98149,  'AL':27843,  'SE':21918,  'BA':564830,
    'MG':586521, 'ES':46095,  'RJ':43750,  'SP':248219,
    'PR':199315, 'SC':95703,  'RS':281748, 'MS':357145,
    'MT':903366, 'GO':340111, 'DF':5760
}

print('Dicionários prontos!')

In [ ]:
# Adicionar população e área ao DataFrame que veio da API do IBGE
df_estados['Populacao_2022'] = df_estados['Sigla'].map(populacao_censo2022)
df_estados['Area_km2']       = df_estados['Sigla'].map(area_km2)
df_estados['Densidade']      = (df_estados['Populacao_2022'] / df_estados['Area_km2']).round(2)

# Esse é o nosso DataFrame principal — estrutura da API IBGE + dados do Censo 2022
df = df_estados.copy()

print(f'DataFrame completo: {df.shape[0]} estados, {df.shape[1]} colunas')
df.head()

In [ ]:

# População total do Brasil
total = df['Populacao_2022'].sum()
print(f'Populacao total do Brasil — Censo 2022: {total:,}')


In [ ]:
# Top 10 estados mais populosos
top10 = df.sort_values('Populacao_2022', ascending=True).tail(10)

plt.figure(figsize=(10, 6))
plt.barh(top10['Estado'], top10['Populacao_2022'] / 1_000_000, color='steelblue')
plt.xlabel('Populacao (milhoes)')
plt.title('Top 10 Estados Mais Populosos — Censo IBGE 2022')
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico: população por região
pop_regiao = df.groupby('Regiao')['Populacao_2022'].sum().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
plt.bar(pop_regiao.index, pop_regiao.values / 1_000_000,
        color=['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0'])
plt.ylabel('Populacao (milhoes)')
plt.title('Populacao por Regiao do Brasil — IBGE 2022')
plt.tight_layout()
plt.show()

In [ ]:

# Top 5 com maior e menor densidade
print('Top 5 — MAIOR densidade (hab/km²):')
print(df.sort_values('Densidade', ascending=False).head(5)[['Estado', 'Sigla', 'Densidade']])
print()
print('Top 5 — MENOR densidade (hab/km²):')
print(df.sort_values('Densidade').head(5)[['Estado', 'Sigla', 'Densidade']])


In [ ]:
# Salvar o DataFrame como CSV
df.to_csv('./dados/populacao_brasil_ibge2022.csv', index=False, encoding='utf-8-sig')
print('Arquivo salvo em: ./dados/populacao_brasil_ibge2022.csv')


---
## 🕑 PARTE 3 — Kaggle: O que é e como usar (20h30 – 21h)

### O que é o Kaggle?

O **Kaggle** (kaggle.com) é a maior plataforma de ciência de dados do mundo. Você pode:
- Baixar **milhares de datasets gratuitos**
- Participar de **competições** de Machine Learning
- Aprender com **notebooks** de outros usuários

### Como baixar um dataset do Kaggle?

#### Opção 1 — Pelo site (mais simples):
1. Acesse **`kaggle.com`** e crie uma conta gratuita
2. Pesquise um dataset (ex: `brazil houses`, `ENEM`, `futebol`)
3. Clique em **Download** (ícone de seta)
4. Extraia o ZIP e coloque o CSV na pasta `dados/`
5. Carregue com `pd.read_csv('./dados/nome_do_arquivo.csv')`

#### Opção 2 — Pela API do Kaggle:
```bash
pip install kaggle
kaggle datasets download -d rubenssjr/brasilian-houses-to-rent
```

### Datasets recomendados para iniciantes:

| Dataset | Tema | URL no Kaggle |
|---------|------|---------------|
| Brazilian Houses to Rent | Imóveis BR | kaggle.com/datasets/rubenssjr/brasilian-houses-to-rent |
| Titanic | Clássico iniciante | kaggle.com/competitions/titanic |
| Netflix Movies | Entretenimento | kaggle.com/datasets/shivamb/netflix-shows |

### Usando o Titanic sem precisar baixar

O dataset Titanic está disponível online e pode ser carregado direto com `pd.read_csv(url)` — sem precisar do Kaggle!


In [ ]:

# Carregar o Titanic direto da internet — sem seaborn, sem baixar arquivo
url_titanic = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'

titanic = pd.read_csv(url_titanic)

print('Dataset Titanic carregado!')
print(f'Linhas: {titanic.shape[0]} | Colunas: {titanic.shape[1]}')
titanic.head()


In [ ]:
# Verificar valores nulos
print('Valores nulos por coluna:')
titanic.isnull().sum()

In [ ]:

# Taxa de sobrevivência geral
sobreviventes = titanic['Survived'].value_counts()
print(f'Nao sobreviveu: {sobreviventes[0]}')
print(f'Sobreviveu:     {sobreviventes[1]}')
print(f'Taxa de sobrevivencia: {sobreviventes[1] / len(titanic) * 100:.1f}%')


In [ ]:

# Sobrevivência por sexo
sobrev_sexo = titanic.groupby('Sex')['Survived'].mean() * 100

plt.figure(figsize=(6, 4))
plt.bar(sobrev_sexo.index, sobrev_sexo.values, color=['salmon', 'steelblue'])
plt.ylabel('Taxa de Sobrevivencia (%)')
plt.title('Titanic — Sobrevivencia por Sexo')
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

# Sobrevivência por classe
sobrev_classe = titanic.groupby('Pclass')['Survived'].mean() * 100

plt.figure(figsize=(6, 4))
plt.bar(['1a Classe', '2a Classe', '3a Classe'], sobrev_classe.values,
        color=['gold', 'silver', '#cd7f32'])
plt.ylabel('Taxa de Sobrevivencia (%)')
plt.title('Titanic — Sobrevivencia por Classe')
plt.ylim(0, 100)
plt.tight_layout()
plt.show()


In [ ]:
# Salvar como CSV (simulando o fluxo do Kaggle)
titanic.to_csv('./dados/titanic.csv', index=False)
print('Dataset salvo em: ./dados/titanic.csv')

---
## 🕒 PARTE 4 — Prática: IDEB por Estado (21h – 22h)

Agora praticamos o fluxo completo com dados do **INEP**:

> **Fonte:** INEP — Índice de Desenvolvimento da Educação Básica  
> **Portal:** `https://www.gov.br/inep/pt-br/areas-de-atuacao/pesquisas-estatisticas-e-indicadores/ideb`  
> **Download direto:** `https://download.inep.gov.br/ideb/resultados/divulgacao_regioes_ufs_ideb_2023.xlsx`

In [ ]:
# Dataset IDEB por estado — valores reais (INEP 2023)
dados_ideb = {
    'Sigla': ['SP','MG','ES','RS','SC','PR','GO','MT','MS','DF',
              'RJ','CE','PE','BA','MA','PI','RN','PB','AL','SE',
              'PA','AM','RO','AC','AP','RR','TO'],
    'IDEB_Anos_Iniciais': [6.8,6.5,6.7,6.5,7.0,6.5,6.2,5.9,6.0,7.0,
                           5.8,5.6,5.3,4.7,4.5,5.1,5.3,5.0,4.5,5.2,
                           4.5,4.6,5.3,5.0,4.7,5.0,5.2],
    'IDEB_Anos_Finais':   [5.2,5.1,5.4,5.3,5.8,5.3,5.0,4.9,5.0,5.8,
                           4.5,4.7,4.3,3.8,3.5,4.0,4.2,4.0,3.6,4.1,
                           3.7,3.8,4.3,4.0,3.8,4.1,4.2]
}

df_ideb = pd.DataFrame(dados_ideb)

# Mesclar com o DataFrame da API do IBGE (que já tem Estado e Regiao)
df_ideb = df_ideb.merge(df[['Sigla', 'Estado', 'Regiao']], on='Sigla', how='left')

print('Dataset IDEB criado e mesclado com dados da API IBGE!')
df_ideb.head()

In [ ]:
# Média do IDEB por região
media_regiao = df_ideb.groupby('Regiao')[['IDEB_Anos_Iniciais', 'IDEB_Anos_Finais']].mean().round(2)
print('Média do IDEB por Região:')
media_regiao

In [ ]:

# Gráfico comparativo por região
media_regiao.plot(kind='bar', figsize=(10, 5), color=['steelblue', 'orange'])
plt.xticks(rotation=20)
plt.ylabel('Nota IDEB')
plt.title('Media do IDEB por Regiao do Brasil | Fonte: INEP')
plt.legend(['Anos Iniciais', 'Anos Finais'])
plt.ylim(0, 8)
plt.tight_layout()
plt.show()



---
## ✏️ Exercícios

**Exercício 1:** Usando `df` (dados da API IBGE + Censo 2022), filtre os estados do **Norte** e mostre qual é o mais populoso.

**Exercício 2:** Com o `df_ipca` (API do Banco Central), mostre o IPCA acumulado de cada ano.

**Exercício 3:** Crie um gráfico de pizza com a **proporção da população por região** do Brasil.

**Exercício 4:** Acesse **dados.gov.br**, escolha um dataset de seu interesse, copie o link do CSV e carregue com `pd.read_csv(url)`.

**Exercício 5 (desafio):** Mescle `df` com `df_ideb` e crie um gráfico de dispersão mostrando se estados mais populosos têm IDEB mais alto ou mais baixo.


In [ ]:

# Exercício 1 — Resolução
norte = df[df['Regiao'] == 'Norte'].sort_values('Populacao_2022', ascending=False)
print('Estados do Norte por populacao:')
print(norte[['Estado', 'Sigla', 'Populacao_2022']])
print(f'\nMais populoso: {norte["Estado"].head(1).values[0]}')


In [ ]:

# Exercício 2 — Resolução
ipca_por_ano = df_ipca.groupby('Ano')['valor'].sum().round(2)

print('IPCA acumulado por ano (%):')
print(ipca_por_ano)


In [ ]:
# Exercício 3 — Resolução
pop_regiao = df.groupby('Regiao')['Populacao_2022'].sum()
plt.figure(figsize=(7, 7))
plt.pie(pop_regiao.values, labels=pop_regiao.index, autopct='%1.1f%%', startangle=90)
plt.title('Proporcao da Populacao por Regiao — Brasil 2022')
plt.tight_layout()
plt.show()

In [ ]:

# Exercício 5 — Resolução
df_merge = df.merge(df_ideb[['Sigla', 'IDEB_Anos_Iniciais']], on='Sigla', how='left')

plt.figure(figsize=(8, 5))
plt.scatter(df_merge['Populacao_2022'] / 1_000_000, df_merge['IDEB_Anos_Iniciais'],
            alpha=0.6, color='steelblue', s=80)
plt.xlabel('Populacao (milhoes)')
plt.ylabel('IDEB Anos Iniciais')
plt.title('Populacao vs. IDEB por Estado')
plt.tight_layout()
plt.show()



---
## 📋 Resumo do Dia 1

Hoje aprendemos:
- ✅ O que é uma **API** e por que usá-la em vez de baixar arquivos
- ✅ **API do IBGE** (`servicodados.ibge.gov.br`) — `requests.get()` + laço `for` para montar DataFrame
- ✅ **API do Banco Central** (`api.bcb.gov.br`) — IPCA com `requests.get()` + `pd.DataFrame()`
- ✅ **dados.gov.br** — como navegar e carregar CSVs diretamente pela URL
- ✅ **Kaggle** — como encontrar e carregar datasets com `pd.read_csv(url)`
- ✅ Mesclar DataFrames de fontes diferentes com `.merge()`
- ✅ Explorar e visualizar dados reais com pandas e matplotlib

> **Dica:** Sempre use `requests.get()` quando a API retornar erro com `pd.read_json(url)`.  
> A diferença é que `requests` permite adicionar **headers** que algumas APIs exigem.

## 🔜 Próxima aula (28/05)
**Web Scraping** — quando não existe API nem CSV, coletamos os dados diretamente do HTML do site!
